# Explore option scenarios

MorphIQ SDK for Python

Explore an absolute underlying move or an explicit spot/volatility grid. Five means
five quote-currency units, not five percent. Volatility/rate/yield controls use
percent; the request sends decimals. Each point starts from the same base.
The base and 1–99 points must fit your account's price-batch limit. Grid order
is spots first, then volatilities, including duplicates. Signed quantity and
multiplier scale base-relative P&L; fees, financing and trade-entry cost are excluded.
Evaluation uses the existing price rate for the base and every point.

Requires the scenario API deployment and matching account permissions. This SDK
candidate is held from merge/release until API deployment is verified.

Inputs start with synthetic values. Run all cells to display controls: setup
and changing controls send **zero API requests**. Quote sends one free utility
request. Execute sends one billable request, capped by Max CU, after a current
quote. Actions run sequentially. Quotes reserve nothing. Changes invalidate the
quote and displayed result; prior receipts remain in `history`.

A missing charge is **not reported**, never zero. No automatic retries are made.
Enter a key at the masked prompt or set `MORPHIQ_API_KEY`; set
`MORPHIQ_API_BASE_URL` to a controlled service when testing. Never save credentials
or outputs. The widget ceiling is limited to exact JavaScript integers; the
Python client accepts the full unsigned 64-bit range.


In [ ]:
import os
from getpass import getpass

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from morphiq_sdk import (
    APIError,
    MorphIQ,
    PriceRequest,
    ResponseError,
    ScenarioPoint,
    ScenarioPosition,
    ScenarioRequest,
    TransportError,
)

api_key = os.environ.get("MORPHIQ_API_KEY") or getpass("MorphIQ API key: ")
model = widgets.Dropdown(
    options=[("BSM · European", "bsm"), ("BS1993 · American", "bs1993")], description="Model"
)
option_type = widgets.Dropdown(options=["call", "put"], description="Option")
spot = widgets.FloatText(value=100.0, description="Spot")
strike = widgets.FloatText(value=100.0, description="Strike")
expiry = widgets.FloatText(value=1.0, description="Years left")
volatility = widgets.FloatText(value=20.0, description="Volatility %")
rate = widgets.FloatText(value=5.0, description="Rate %")
yield_input = widgets.FloatText(value=0.0, description="Yield %")
multiplier = widgets.FloatText(value=100.0, description="Multiplier")
quote_button = widgets.Button(
    description="Quote · 1 request / 0 CU", layout=widgets.Layout(width="240px")
)
run_button = widgets.Button(
    description="Execute · 1 request", disabled=True, layout=widgets.Layout(width="240px")
)
output = widgets.Output()


def show_message(*values):
    output.append_stdout(" ".join(str(value) for value in values) + "\n")


usage = widgets.HTML(value="No API requests sent.")
history = []
last_quote = None
last_response = None
quote_snapshot = None
result_snapshot = None


def contract_inputs():
    return PriceRequest(
        exercise_style="american" if model.value == "bs1993" else "european",
        pricing_model="bjerksund_stensland_1993"
        if model.value == "bs1993"
        else "black_scholes_merton",
        option_type=option_type.value,
        spot=spot.value,
        strike=strike.value,
        time_to_expiry_years=expiry.value,
        volatility=volatility.value / 100,
        risk_free_rate=rate.value / 100,
        dividend_yield=yield_input.value / 100,
    )


quantity = widgets.FloatText(value=1.0, description="Quantity")
mode = widgets.Dropdown(
    options=[("Single move", "point"), ("Spot × volatility grid", "grid")], description="Scenario"
)
spot_change = widgets.FloatText(value=5.0, description="Spot move")
grid_spots = widgets.Text(value="95,100,105", description="Grid spots")
grid_vols = widgets.Text(value="15,20,25", description="Grid vols %")
max_cu = widgets.BoundedIntText(value=10, min=0, max=2**53 - 1, description="Max CU")
controls = [
    model,
    option_type,
    spot,
    strike,
    expiry,
    volatility,
    rate,
    yield_input,
    quantity,
    multiplier,
    mode,
    spot_change,
    grid_spots,
    grid_vols,
]


def current_request():
    points = [ScenarioPoint(spot_change=spot_change.value)]
    if mode.value == "grid":
        spots = [float(value.strip()) for value in grid_spots.value.split(",")]
        vols = [float(value.strip()) / 100 for value in grid_vols.value.split(",")]
        points = [ScenarioPoint(spot=s, volatility=v) for s in spots for v in vols]
    return ScenarioRequest(
        contract=contract_inputs(),
        scenarios=points,
        position=ScenarioPosition(quantity=quantity.value, multiplier=multiplier.value),
    )


def execute_request(client, request):
    return client.evaluate_scenarios(request, max_cu=max_cu.value)


def show_result(response):
    show_message("Base option price:", response.base.result.price)
    rows = []
    for item in response.results:
        row = {
            "Index": item.index,
            "Spot": item.inputs.spot,
            "Volatility": item.inputs.volatility,
            "Status": item.status,
        }
        if item.status == "success":
            row.update(
                Price=item.result.price, Change=item.price_change, PositionPnL=item.position_pnl
            )
        else:
            row.update(Error=item.error.code)
        rows.append(row)
    output.append_display_data(pd.DataFrame(rows))
    show_message("Prices are per underlying unit; position P&L is relative to the base mark.")
    show_message("Errors remain in submitted order; one receipt covers the complete request.")

In [ ]:
def record(action, request, charged, request_id):
    history.append(
        {"action": action, "request": request, "cu_charged": charged, "request_id": request_id}
    )
    known = sum(row["cu_charged"] for row in history if row["cu_charged"] is not None)
    unknown = sum(row["cu_charged"] is None for row in history)
    usage.value = (
        f"Recorded receipts: {known} CU; {unknown} unknown charges; {len(history)} requests."
    )


def changed(_change):
    global last_quote, last_response, quote_snapshot, result_snapshot
    last_quote = last_response = quote_snapshot = result_snapshot = None
    run_button.disabled = True
    with output:
        output.clear_output(wait=True)
        show_message(
            "Inputs changed. Quote again before execution. Previous receipts remain in history."
        )


for control in controls:
    control.observe(changed, names="value")


def perform(action):
    global last_quote, last_response, quote_snapshot, result_snapshot
    with output:
        output.clear_output(wait=True)
        try:
            request = current_request()
            wire = request.to_wire()
            if action == "execute":
                if last_quote is None or wire != quote_snapshot:
                    show_message("Quote the current inputs first. No execution submitted.")
                    return
                if last_quote.quote.total_fcu > max_cu.value:
                    show_message("Quote exceeds Max CU. No execution submitted.")
                    return
            else:
                last_quote = quote_snapshot = None
            last_response = result_snapshot = None
            for control in controls + [max_cu, quote_button, run_button]:
                control.disabled = True
            with MorphIQ(api_key=api_key) as client:
                if action == "quote":
                    response = client.quote(request)
                    last_quote, quote_snapshot = response, wire
                    record(
                        action, wire, response.metadata.fcu_charged, response.metadata.request_id
                    )
                    show_message(f"Quoted execution charge: {response.quote.total_fcu} CU")
                    show_message("Quote charge:", response.metadata.fcu_charged)
                    show_message(
                        "Quotes reserve nothing and do not lock policy or guarantee funds/capacity."
                    )
                else:
                    response = execute_request(client, request)
                    last_response, result_snapshot = response, wire
                    record(
                        action, wire, response.metadata.fcu_charged, response.metadata.request_id
                    )
                    last_quote = quote_snapshot = None
                    show_message(
                        "Commercial CU charged:",
                        response.metadata.fcu_charged
                        if response.metadata.fcu_charged is not None
                        else "not reported",
                    )
                    show_message("Request ID:", response.metadata.request_id)
                    show_result(response)
        except (APIError, ResponseError, TransportError) as error:
            metadata = getattr(error, "metadata", None)
            charged = 0 if action == "quote" else (metadata.fcu_charged if metadata else None)
            record(action, wire, charged, getattr(error, "request_id", None))
            last_quote = quote_snapshot = None
            show_message(str(error))
            show_message(
                "Commercial CU charged:", charged if charged is not None else "not reported"
            )
            show_message("No retry. A missing response is not evidence of zero charge.")
        except ValueError as error:
            show_message(f"Invalid local inputs/configuration: {error}. No request submitted.")
        finally:
            for control in controls + [max_cu, quote_button]:
                control.disabled = False
            run_button.disabled = last_quote is None


quote_button.on_click(lambda _button: perform("quote"))
run_button.on_click(lambda _button: perform("execute"))
display(widgets.VBox(controls + [max_cu, widgets.HBox([quote_button, run_button]), usage, output]))